<a href="https://colab.research.google.com/github/Raquelsantosb/analise-dados-python/blob/main/Pre_Processamento_e_Analise_Iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio
import seaborn as sb
import matplotlib.pyplot as plt

pio.renderers.default = "colab"

from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

In [39]:
iris = load_iris()
def label(x):
    labels = iris.target_names
    return labels[x]
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df['class_codes'] = iris.target
iris_df['class'] = iris_df['class_codes'].apply(lambda x: label(x))

In [40]:
for c in iris_df.columns:
  boxfig=px.box(iris_df, x=c, color='class', title='Boxplot of '+c)
  boxfig.show()
  histfig=px.histogram(iris_df, x=c, color='class', title='Histogram of '+c)
  histfig.show()

In [41]:
from sklearn.preprocessing import MinMaxScaler
num_columns = iris_df.select_dtypes(include='number')
to_standardize = num_columns.columns.tolist()
to_standardize.remove('class_codes')
scaler = MinMaxScaler((0,1))
iris_df[to_standardize] = scaler.fit_transform(iris_df[to_standardize])

In [42]:
fig = px.scatter_matrix(iris_df, color='class')
fig.show()

In [43]:
pca_comp=2
pca = PCA(n_components=pca_comp)
feature_columns = [ c for c in iris_df.columns if c not in ['class','class_codes']]
pca_iris_df = iris_df[feature_columns]
pca_iris_df = pd.DataFrame(pca.fit_transform(pca_iris_df), columns=['pc_'+str(i) for i in range(0,pca_comp)])
pca_iris_df['class_codes'] = iris_df['class_codes']

In [44]:
# Treinamento do modelo com o dataset Iris reduzido e regularizado

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [45]:
feature_columns = [c for c in pca_iris_df.columns if c != 'class_codes']
X_train, X_test, y_train, y_test = train_test_split(
    pca_iris_df[feature_columns],
    pca_iris_df['class_codes'],
    test_size=0.2,
    random_state=0
)

In [46]:
clf = RandomForestClassifier(max_depth=2, random_state=0)
clf.fit(X_train, y_train)
print(clf.score(X_test, y_test))

0.9333333333333333


In [47]:
clf.feature_importances_

array([0.67391862, 0.32608138])

In [48]:
# validação cruzada utiliza a medida F para medir o modelo.

from sklearn.model_selection import cross_val_score
cv_result = cross_val_score(clf, pca_iris_df[feature_columns],
                            pca_iris_df['class_codes'], cv=5,scoring='f1_weighted')
print("CV Result: ", cv_result.round(3))
print("CV Mean: ", round(cv_result.mean(), 3))
print("CV Std: ", round(cv_result.std(),3))

CV Result:  [0.865 0.933 0.933 0.9   1.   ]
CV Mean:  0.926
CV Std:  0.045


In [49]:
from sklearn.metrics import confusion_matrix
y_true = y_test.values
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_true, y_pred)

In [50]:
x_labels = ['Pred '+str(c) for c in pca_iris_df['class_codes'].unique()]
y_labels = ['True '+str(c) for c in pca_iris_df['class_codes'].unique()]
px.imshow(
    cm,
    labels=dict(x="Predictions", y="Ground Truth"),
    x=x_labels,
    y=y_labels
)

# Matriz de confusão. Gerado pela demonstração em Python

In [51]:
# Extração das metricas do modelo com o sklearn

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
f1 = f1_score(y_true, y_pred, average='weighted')
pr = precision_score(y_true, y_pred, average='weighted')
re = recall_score(y_true, y_pred, average='weighted')
accr = accuracy_score(y_true, y_pred)
metrics = {
    "f_measure": round(f1,3),
    "precision": round(pr,3),
    "recall": round(re,3),
    "accuracy": round(accr,3)
}
print(metrics)

{'f_measure': 0.935, 'precision': 0.95, 'recall': 0.933, 'accuracy': 0.933}


In [52]:
import pickle
# salvar o classificador
with open('my_dumped_classifier.pkl', 'wb') as fid:
  pickle.dump(clf, fid)
# carrega-lo novamente
with open('my_dumped_classifier.pkl', 'rb') as fid:
  clf_loaded = pickle.load(fid)

clf_loaded.predict(X_test[0:1])

array([2])